# Caretta Lab MOKE Testing Notebook

Written by Geo Fratian on 11/12/2026

This python notebook provides an interactive interface with the Magneto-Optic Kerr Effect (MOKE) measurement functionality available in the open source Python Integrated Experiment Control (piec) package. The MOKE measurement uses a field controller, a digital multimeter (DMM), and an optional gaussmeter to sweep magnetic field along a closed bipolar triangle waveform (`0 -> +H_max -> 0 -> -H_min -> 0`) and record optical hysteresis loops. Code documentation and setup details can be found at https://piec.readthedocs.io.

*COPY THIS NOTEBOOK AND RUN IN A LOCAL DIRECTORY TO PREVENT IT FROM BEING OVERWRITTEN BY GITHUB*

### Imports

In [ ]:
from piec.drivers.field_controller.virtual_field_controller import VirtualFieldController
from piec.drivers.dmm.virtual_dmm import VirtualDMM
from piec.drivers.gaussmeter.virtual_gaussmeter import VirtualGaussmeter
from piec.simulation.magnetic_material import ArctanMOKE
from piec.measurement.moke import MOKE
from piec.analysis.utilities import standard_csv_to_metadata_and_data
from piec.drivers.utilities import PiecManager

import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
import os

In [ ]:
pm = PiecManager()
pm.list_resources()

### Instrument Setup

Every MOKE experiment uses two primary instruments and one optional field monitor:
- **Field Controller** (`field_controller`): Commands or ramps the applied magnetic field in Oersted (Oe).
- **Digital Multimeter** (`dmm`): Reads the optical detector (photodiode / balanced bridge) voltage in Volts (V).
- **Gaussmeter** (`gaussmeter`, optional): Reads back the actual magnetic field in Oersted (Oe) (`None` uses the field controller's calibrated setpoint).

In `piec`, each instrument is represented by a class object. Initialize them with the appropriate VISA address, or use `'VIRTUAL'` addresses to run in simulation mode with an ideal `ArctanMOKE` sample without real hardware. Commented-out examples below also show how to use the DAQ emulators (`DaqFieldController`, `DaqAsDMM`, `DaqAsGaussmeter`) when driving the setup through a shared DAQ.

In [ ]:
# --- Standard Virtual / Physical Instrument Setup ---
sample = ArctanMOKE(coercivity=100.0, switching_width=30.0, bias=0.0)

field_controller = VirtualFieldController('VIRTUAL')  # Field Controller
field_controller.mag_sample = sample

dmm = VirtualDMM('VIRTUAL', voltage_reader=sample.get_moke_voltage)  # Optical DMM: e.g. 'GPIB0::22::INSTR' for real
gaussmeter = VirtualGaussmeter('VIRTUAL')  # Optional Gaussmeter (or set gaussmeter = None)

# --- Optional: DAQ Emulator Setup (Virtual or Physical MCC DAQ) ---
# from piec.drivers.daq.usb1208hs import USB1208HS
# from piec.drivers.emulators.daq_to_field_controller import DaqFieldController
# from piec.drivers.emulators.daq_to_dmm import DaqAsDMM
# from piec.drivers.emulators.daq_to_gaussmeter import DaqAsGaussmeter
#
# daq = USB1208HS('VIRTUAL')         # Use 'VIRTUAL' for simulation, or address=0 for physical MCC DAQ
# field_controller = DaqFieldController(daq, ao_channel=0, gain=100.0)  # AO0 -> magnet supply (100 Oe/V)
# dmm = DaqAsDMM(daq, ai_channel=2)                                     # AI2 -> optical detector
# gaussmeter = DaqAsGaussmeter(daq, ai_channel=0, gain=100.0)           # AI0 -> field monitor (or None)

print('Field Controller: ', field_controller.idn())
print('DMM: ', dmm.idn())
if gaussmeter is not None:
    print('Gaussmeter: ', gaussmeter.idn())

### MOKE Measurement

The MOKE measurement sweeps the applied magnetic field through a closed bipolar triangle cycle (`0 -> +field_max -> 0 -> field_min -> 0`) while measuring the optical detector voltage via the DMM.

Key parameters:
- **geometry**: Optical MOKE configuration (`'longitudinal'`, `'polar'`, or `'transverse'`).
- **mode**: `'stepped'` for discrete field steps or `'continuous'` for timed triangle sweeps.
- **field_min / field_max**: Minimum and maximum applied magnetic field in Oersted (Oe).
- **points_per_cycle**: Number of field steps per cycle in `'stepped'` mode, and the averaging grid size in `'continuous'` mode.
- **cycles**: Number of measured hysteresis loops to average.
- **discard_cycles**: Number of initial conditioning cycles to exclude from the cycle average.
- **dwell_time**: Settling delay at each step in `'stepped'` mode (seconds).
- **cycle_period / sample_rate**: Sweep period (s) and acquisition rate (Hz) that determine raw sample count per cycle in `'continuous'` mode.

In [ ]:
### VARIABLE DEFINITIONS ###

path = r"scratch"           # path to save data, make sure directory exists and you have permissions

geometry = 'longitudinal'   # optical geometry: 'longitudinal', 'polar', or 'transverse'
mode = 'stepped'            # acquisition mode: 'stepped' or 'continuous'
field_min = -500.0          # minimum magnetic field (Oe)
field_max = 500.0           # maximum magnetic field (Oe)
points_per_cycle = 200      # points per full cycle (0 -> +H_max -> 0 -> -H_min -> 0)
cycles = 3                  # number of measured cycles to average
discard_cycles = 1          # initial conditioning cycles to discard before averaging
dwell_time = 0.001          # wait time per point in stepped mode (s)
cycle_period = 1.0          # period per cycle in continuous mode (s)
sample_rate = 200.0         # sampling rate in continuous mode (Hz)

In [ ]:
### RUN ###

experiment = MOKE(
    field_controller=field_controller,
    dmm=dmm,
    gaussmeter=gaussmeter,
    geometry=geometry,
    mode=mode,
    field_min=field_min,
    field_max=field_max,
    points_per_cycle=points_per_cycle,
    cycles=cycles,
    discard_cycles=discard_cycles,
    dwell_time=dwell_time,
    cycle_period=cycle_period,
    sample_rate=sample_rate,
    save_dir=path,
)

# Run the experiment — this sweeps the magnetic field, records the optical
# hysteresis loop, computes the cycle average, zeroes the field, and saves CSVs.
experiment.run_experiment()

In [ ]:
# Retrieve the captured raw and cycle-averaged data for further analysis
moke_metadata, moke_df = standard_csv_to_metadata_and_data(experiment.filename)
avg_metadata, avg_df = standard_csv_to_metadata_and_data(experiment.average_filename)
avg_df.head()

In [ ]:
# Plot the MOKE hysteresis loop: Optical Signal vs Magnetic Field
avg_df.plot(x="field (Oe)", y="signal (V)", figsize=(8, 5), marker=".", color="k", legend=False)
plt.xlabel("Magnetic Field (Oe)")
plt.ylabel("Optical Signal (V)")
plt.title("MOKE Hysteresis Loop (Cycle Average)")
plt.grid(True, linestyle="--", alpha=0.3)
plt.tight_layout()
plt.show()